<a href="https://colab.research.google.com/github/Alltingzrpozz/Alltingzrpozz/blob/main/Natural_Language_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries and Data Loader

In [5]:
import pandas as pd
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# Download necessary NLTK resources
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab') # Added to resolve the LookupError

# Load datasets
df_train = pd.read_csv('/content/sample_data/train.csv')
df_test = pd.read_csv('/content/sample_data/test.csv')

print(f"Training rows: {len(df_train)}, Test rows: {len(df_test)}")

Training rows: 13240, Test rows: 460


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Data Exploration & Splitting

In [6]:
# Understanding the label distribution
print(df_train['label'].value_counts())

# 70/30 Split
X_train, X_val, y_train, y_val = train_test_split(
    df_train['tweet'],
    df_train['label'],
    test_size=0.3,
    random_state=100
)

label
NOT    8840
TIN    3876
UNT     524
Name: count, dtype: int64


# Data Preprocessing

In [7]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_tweet(text):
    # 1. Remove URLs and @User mentions (Noise Reduction)
    text = re.sub(r'http\S+|www\S+|@\w+', '', text)
    # 2. Lower-casing (Standardisation)
    text = text.lower()
    # 3. Remove non-ASCII characters (Emojis/Special symbols)
    text = text.encode('ascii', 'ignore').decode('ascii')
    # 4. Tokenisation & Lemmatisation (Linguistic Units)
    tokens = nltk.word_tokenize(text)
    # 5. Remove Stop Words (Focus on Signal)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and t.isalnum()]

    return " ".join(tokens)

# Apply cleaning to training and validation sets
X_train_clean = X_train.apply(clean_tweet)
X_val_clean = X_val.apply(clean_tweet)

# Feature Extraction

In [8]:
# Optimisation: Using unigrams and bigrams
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)

# Fit on training and transform both
X_train_tfidf = vectorizer.fit_transform(X_train_clean)
X_val_tfidf = vectorizer.transform(X_val_clean)

# Model Training and Evaluation

In [9]:
# Initialize and train the model
model = LinearSVC(random_state=100)
model.fit(X_train_tfidf, y_train)

# Predict on validation set
val_predictions = model.predict(X_val_tfidf)

# Critical Evaluation: Performance Report
print(classification_report(y_val, val_predictions))

              precision    recall  f1-score   support

         NOT       0.78      0.87      0.82      2638
         TIN       0.59      0.49      0.54      1177
         UNT       0.26      0.04      0.07       157

    accuracy                           0.73      3972
   macro avg       0.54      0.47      0.48      3972
weighted avg       0.70      0.73      0.71      3972



# Model Predictions

In [14]:
# 1. Preprocess the test data using the exact same function
df_test['clean_tweet'] = df_test['tweet'].apply(clean_tweet)

# 2. Convert to features using the SAME vectorizer (transform only)
X_test_tfidf = vectorizer.transform(df_test['clean_tweet'])

# 3. Make predictions using the best trained model
df_test['prediction'] = model.predict(X_test_tfidf)

# 4. View a sample of n=5 results for critical evaluation
print("Sample of 5 Predictions for Audit:")
display(df_test[['id', 'clean_tweet', 'prediction']].sample(5))

# 5. SAVE to CSV including the cleaned tweet column
df_test[['id', 'clean_tweet', 'prediction']].to_csv('test-predictions.csv', index=False)

print("\nSuccess: 'test-predictions.csv' created with id, clean_tweet, and prediction.")

Sample of 5 Predictions for Audit:


,id,clean_tweet,prediction
373,77746,death want death metal tell hateful keebler el...,TIN
25,83155,hiac damn matt hardy randy orton put one hell ...,NOT
260,80397,liberal reaching peak desperation call phillip...,NOT
163,13959,wce favorite girl hope day awesome url,NOT
274,51948,walkawayfromcatholicism corrupt core become gi...,TIN



Success: 'test-predictions.csv' created with id, clean_tweet, and prediction.
